### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [25]:
!pip install scikeras

In [26]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
#from keras.wrappers.scikit_learn import KerasClassifier
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [27]:
data=pd.read_csv("dataset/Churn_Modelling.csv")

In [28]:
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [29]:
gender = LabelEncoder()
data["Gender"] = gender.fit_transform(data["Gender"])


from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder()
geo_encoder= onehot_encoder.fit_transform(data[["Geography"]])

geo_encoded_df = pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder.get_feature_names_out(["Geography"]))

data = pd.concat([data.drop('Geography', axis=1),geo_encoded_df],axis=1)

x= data.drop("Exited",axis=1)
y=data["Exited"]

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# save the encoders and scalers
with open("models/hyper_label_encoder_gender.pkl","wb") as f:
    pickle.dump(gender,f)

with open("models/hyper_onehot_encoder_geo.pkl","wb") as f:
    pickle.dump(onehot_encoder,f)

with open("models/hyper_scaler.pkl","wb") as f:
    pickle.dump(scaler,f)

In [ ]:
def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(x_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model


In [31]:
## Create a keras classifier 
model=KerasClassifier(layers=1,neurons=32,build_fn=create_model,epochs=50, batch_size=10, verbose=0)

In [32]:
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2, 3],
    'epochs': [50, 100]
}



In [33]:
print(x_train.shape, y_train.shape)


(8000, 12) (8000,)


In [34]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1,cv=3, verbose=1)
grid_result = grid.fit(x_train, y_train)
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))


Fitting 3 folds for each of 24 candidates, totalling 72 fits


d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\02-Machine_Learning_For_NLP\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
d:\Udemy_Generative_AI_course\Enterprise_AI_Engineer_RoadMap\02-Machine_Learning_For_NLP\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best: 0.855749 using {'epochs': 50, 'layers': 1, 'neurons': 16}
